In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from pathlib import Path
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPreTokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

c:\Users\Dmytro\Desktop\python\UkrainianSmallTransformer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
TOKENIZER_VOCAB_SIZE = 16000
TOKENIZER_DOCS = 100000

DATA_DIR = Path("data")
TOKENIZER_DIR = Path("tokenizer")

DATA_DIR.mkdir(exist_ok=True)
TOKENIZER_DIR.mkdir(exist_ok=True)

TRAIN_DATA_PATH = DATA_DIR / "train_500M.bin"

VAL_DATA_DIR = DATA_DIR / "validation"
TEST_DATA_DIR = DATA_DIR / "test"
TEST_DATA_DIR.mkdir(exist_ok=True)
VAL_DATA_DIR.mkdir(exist_ok=True)

VAL_DATA_PATH = VAL_DATA_DIR / "validation_5M.bin"
TEST_DATA_PATH = TEST_DATA_DIR / "test_5M.bin"

TOKENS_100M = 100_000_000
TOKENS_250M = 250_000_000
TOKENS_500M = 500_000_000
VAL_TOKENS = 5_000_000
TEST_TOKENS = 5_000_000


In [3]:
data = load_dataset("HuggingFaceFW/fineweb-2", "ukr_Cyrl", split="train", streaming=True)
samples = list(data.take(5))

datapd = pd.DataFrame(samples)

print(datapd["text"].iloc[0])

Рубрика: класичний портрет
Соломія Крушельницька: «Тріумф щовечора… це забагато»23 февраля 2009
Автор Євгенія Вятчанінова
Час і місто народження: 23 вересня 1872 року, с. Білявинці, Тернопілля
Походження: донька священика, онука відомого галицького письменника — Григорія Савчинського
Мистецька Батьківщина: Італія, Віареджо
Голос: драматичне сопрано
Опери: «Аїда», «Трубадур» Д. Верді, «Фауст» Ш. Гуно, «Валькірія», «Лоенгрін», «Трістан та Ізольда» Р. Вагнера, «Страшний двір», «Галька», «Графиня» С. Монюшка, «Африканка» Д. Мейєрбера, «Манон Леско» і «Чіо-Чіо-Сан» Д. Пуччіні, «Кармен» Ж. Бізе, «Електра» Р. Штрауса, «Євгеній Онєгін» і «Пікова дама» П. Чайковського.
Родина: чоловік — Чезаре Річчіоне, дітей не було
Повернення в Україну: 1939 рік, Галичина
Хвороба: рак горла
Смерть: 16 листопада 1952 року, Львів, похована на Личаківському кладовищі
Соломія була рішучою і сміливою, витонченою і ніжною, наївною і цілеспрямованою… Власне, вона була готова до всіх своїх втілень… Брунгільда… Чіо-чі

In [4]:
corpus_path = DATA_DIR / "tokenizer_corpus.txt"

count = 0

with open(corpus_path, "w", encoding="utf-8") as f:
    for row in data:
        text = row["text"].strip()

        if not text:
            continue

        # Зберігаємо структуру тексту (включно з переносами рядків)
        f.write(text + "\n\n")

        count += 1

        if count >= TOKENIZER_DOCS:
            break

print(f"Saved {count} documents")
print(f"Corpus: {corpus_path}")
print(
    f"Corpus size: "
    f"{corpus_path.stat().st_size / 1024**2:.2f} MB"
)
with open(corpus_path, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline()[:500])
        print("-" * 80)


Saved 100000 documents
Corpus: data\tokenizer_corpus.txt
Corpus size: 617.12 MB
Рубрика: класичний портрет

--------------------------------------------------------------------------------
Соломія Крушельницька: «Тріумф щовечора… це забагато»23 февраля 2009

--------------------------------------------------------------------------------
Автор Євгенія Вятчанінова

--------------------------------------------------------------------------------


In [5]:
tokenizer = Tokenizer(BPE(unknown_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevelPreTokenizer()
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(
    vocab_size=TOKENIZER_VOCAB_SIZE,
    min_frequency=1,
    special_tokens=[
        "[UNK]",
        "[BOS]",
        "[EOS]",
        "[PAD]",
    ],
)
tokenizer.train([str(corpus_path)], trainer)

In [6]:
print(f"Tokenizer vocab size: {tokenizer.get_vocab_size()}")
text = "Привіт, я ГПТ українською мовою!"
encoded = tokenizer.encode(text)
decode = tokenizer.decode(encoded.ids)
print("Original: ", text)
print("\nEncoded: ", encoded.tokens)
print("\nDecoded: ", decode)
print("\nNumber of tokens:", len(encoded.tokens))

Tokenizer vocab size: 16000
Original:  Привіт, я ГПТ українською мовою!

Encoded:  ['ĠÐŁÑĢÐ¸', 'Ð²ÑĸÑĤ', ',', 'ĠÑı', 'ĠÐĵ', 'ÐŁ', 'Ð¢', 'ĠÑĥÐºÑĢÐ°ÑĹÐ½ÑģÑĮÐºÐ¾Ñİ', 'ĠÐ¼Ð¾Ð²Ð¾Ñİ', '!']

Decoded:   Привіт, я ГПТ українською мовою!

Number of tokens: 10


In [7]:
tokenizer_path = TOKENIZER_DIR / "tokenizer.json"

tokenizer.save(str(tokenizer_path))

print(f"Tokenizer saved to: {tokenizer_path}")

Tokenizer saved to: tokenizer\tokenizer.json


In [8]:
if TRAIN_DATA_PATH.exists():
    tokens = np.memmap(
        TRAIN_DATA_PATH,
        dtype=np.uint16,
        mode="r+",
        shape=(TOKENS_500M,),
    )
else:
    tokens = np.memmap(
        TRAIN_DATA_PATH,
        dtype=np.uint16,
        mode="w+",
        shape=(TOKENS_500M,),
    )
    
train_data = load_dataset(
    "HuggingFaceFW/fineweb-2",
    "ukr_Cyrl",
    split="train",
    streaming=True,
)

train_data = train_data.skip(TOKENIZER_DOCS)

eos_id = tokenizer.token_to_id("[EOS]")
position = 0
documents = 0
for row in train_data:
    text = row["text"].strip()

    if not text:
        continue

    encoded = tokenizer.encode(text)
    
    ids = encoded.ids

    ids.append(eos_id)

    remaining = TOKENS_500M - position

    ids = ids[:remaining]

    tokens[position:position + len(ids)] = ids

    position += len(ids)
    documents += 1

    if documents % 1000 == 0:
        print(
            f"Документів: {documents:,} | "
            f"Токенів: {position:,} / {TOKENS_500M:,}"
        )

    if position >= TOKENS_500M:
        break
tokens.flush()
print(f"Готово: {position:,} токенів із {documents:,} документів")

Документів: 1,000 | Токенів: 850,294 / 500,000,000
Документів: 2,000 | Токенів: 1,659,516 / 500,000,000
Документів: 3,000 | Токенів: 2,509,568 / 500,000,000
Документів: 4,000 | Токенів: 3,321,480 / 500,000,000
Документів: 5,000 | Токенів: 4,282,264 / 500,000,000
Документів: 6,000 | Токенів: 5,087,493 / 500,000,000
Документів: 7,000 | Токенів: 6,063,874 / 500,000,000
Документів: 8,000 | Токенів: 6,902,001 / 500,000,000
Документів: 9,000 | Токенів: 7,691,439 / 500,000,000
Документів: 10,000 | Токенів: 8,543,341 / 500,000,000
Документів: 11,000 | Токенів: 9,425,538 / 500,000,000
Документів: 12,000 | Токенів: 10,270,222 / 500,000,000
Документів: 13,000 | Токенів: 11,154,217 / 500,000,000
Документів: 14,000 | Токенів: 11,956,455 / 500,000,000
Документів: 15,000 | Токенів: 12,808,422 / 500,000,000
Документів: 16,000 | Токенів: 13,726,220 / 500,000,000
Документів: 17,000 | Токенів: 14,515,785 / 500,000,000
Документів: 18,000 | Токенів: 15,299,213 / 500,000,000
Документів: 19,000 | Токенів: 16

In [9]:
if VAL_DATA_PATH.exists():
    val_tokens = np.memmap(
        VAL_DATA_PATH,
        dtype=np.uint16,
        mode="r+",
        shape=(VAL_TOKENS,),
    )
else:
    val_tokens = np.memmap(
        VAL_DATA_PATH,
        dtype=np.uint16,
        mode="w+",
        shape=(VAL_TOKENS,),
    )
val_data = load_dataset(
    "HuggingFaceFW/fineweb-2",
    "ukr_Cyrl",
    split="train",
    streaming=True,
)
val_data = val_data.skip(TOKENIZER_DOCS + documents)
position = 0
val_documents = 0

for row in val_data:
    text = row["text"].strip()

    if not text:
        continue

    ids = tokenizer.encode(text).ids
    ids.append(eos_id)

    remaining = VAL_TOKENS - position
    ids = ids[:remaining]

    val_tokens[position:position + len(ids)] = ids

    position += len(ids)
    val_documents += 1
    
    if position >= VAL_TOKENS:
            break
    
val_tokens.flush()
print(f"Готово: {position:,} токенів із {val_documents:,} документів")

Готово: 5,000,000 токенів із 5,922 документів


In [10]:
if TEST_DATA_PATH.exists():
    test_tokens = np.memmap(
        TEST_DATA_PATH,
        dtype=np.uint16,
        mode="r+",
        shape=(TEST_TOKENS,),
    )
else:
    test_tokens = np.memmap(
        TEST_DATA_PATH,
        dtype=np.uint16,
        mode="w+",
        shape=(TEST_TOKENS,),
    )
test_data = load_dataset(
    "HuggingFaceFW/fineweb-2",
    "ukr_Cyrl",
    split="train",
    streaming=True,
)
test_data = test_data.skip(TOKENIZER_DOCS + documents + val_documents)
position = 0
test_documents = 0

for row in test_data:
    text = row["text"].strip()

    if not text:
        continue

    ids = tokenizer.encode(text).ids
    ids.append(eos_id)

    remaining = TEST_TOKENS - position
    ids = ids[:remaining]

    test_tokens[position:position + len(ids)] = ids

    position += len(ids)
    test_documents += 1
    
    if position >= TEST_TOKENS:
            break
    
test_tokens.flush()
print(f"Готово: {position:,} токенів із {test_documents:,} документів")

Готово: 5,000,000 токенів із 6,027 документів


In [11]:

print(tokenizer.token_to_id("[UNK]"))
print(tokenizer.token_to_id("[BOS]"))
print(tokenizer.token_to_id("[EOS]"))
print(tokenizer.token_to_id("[PAD]"))

0
1
2
3
